In [ ]:
!rm -rf /kaggle/working/visual_search

!git clone https://github.com/Nihit3003/Visual-Product-Search-Engine \
    /kaggle/working/visual_search

In [ ]:
import warnings

warnings.filterwarnings("ignore")

In [ ]:
import numpy as np
import torchvision
import torch

print(np.__version__)

print("Torchvision OK")

print(
    "CUDA Available:",
    torch.cuda.is_available()
)

if torch.cuda.is_available():

    print(
        "GPU:",
        torch.cuda.get_device_name(0)
    )

In [ ]:
import subprocess
import sys

packages = [

    "hnswlib",

    "open-clip-torch==2.24.0",

    "transformers==4.41.2",

    "accelerate==0.28.0",

    "sentencepiece",

    "ultralytics==8.3.0",

    "timm",

    "einops",
]

for pkg in packages:

    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", pkg],
        check=True
    )

print("Packages installed")

In [ ]:
import os
import sys

PROJECT_ROOT = "/kaggle/working/visual_search"

sys.path.insert(0, PROJECT_ROOT)

# =========================================================
# DATASET
# =========================================================

DATASET_ROOT = (
    "/kaggle/input/"
    "deepfashion-inshop-dataset/"
    "Dataset"
)

# =========================================================
# INPUTS
# =========================================================

CHECKPOINT_DATASET = (
    "/kaggle/input/"
    "clip-finetuned-best"
)

INDEX_DATASET = (
    "/kaggle/input/"
    "YOUR_INDEX_DATASET"
)

# =========================================================
# PATHS
# =========================================================

CKPT_PATH = (
    f"{CHECKPOINT_DATASET}/"
    "clip_finetuned_best.pt"
)

INDEX_DIR = (
    f"{INDEX_DATASET}/"
    "index"
)

RESULTS_DIR = (
    "/kaggle/working/results"
)

os.makedirs(
    RESULTS_DIR,
    exist_ok=True
)

print("Checkpoint:")

print(CKPT_PATH)

print("\nIndex dir:")

print(INDEX_DIR)

In [ ]:
content = """
# Visual Product Search Engine — Source Package

from .dataset import (
    build_dataloaders,
    get_clip_transform,
    parse_eval_partition,
    parse_bboxes,
    infer_category,
    DeepFashionDataset,
    bbox_crop,
)

from .model import (
    VisualSearchModel,
    SupConLoss,
)

from .blip_module import (
    FashionCaptioner,
    ITMReranker,
)

from .localizer import (
    YOLOLocalizer,
)

from .index import (
    HNSWIndex,
)

from .metrics import (
    evaluate,
    MetricResults,
    evaluate_multi_seed,
)
"""

with open(
    "/kaggle/working/visual_search/src/__init__.py",
    "w"
) as f:

    f.write(content)

print("Fixed __init__.py")

In [ ]:
!python /kaggle/working/visual_search/scripts/evaluate.py \
    --dataset_root "{DATASET_ROOT}" \
    --index_base "{INDEX_DIR}" \
    --ckpt_path "{CKPT_PATH}" \
    --output_dir "{RESULTS_DIR}" \
    --embed_dim 768 \
    --batch_size 64 \
    --alpha_B 0.7 \
    --alpha_C 0.7

In [ ]:
import json
import pandas as pd

with open(
    f"{RESULTS_DIR}/ablation_results.json"
) as f:

    res = json.load(f)

rows = []

for cond, metrics in res.items():

    row = {
        "Condition":
        cond.replace(
            "condition_",
            ""
        )
    }

    for metric, vals in metrics.items():

        if not isinstance(vals, dict):
            continue

        if "mean" not in vals:
            continue

        if "std" not in vals:
            continue

        row[metric] = (
            f"{vals['mean']:.4f} ± "
            f"{vals['std']:.4f}"
        )

    rows.append(row)

df = pd.DataFrame(rows)

if not df.empty:

    df = df.set_index(
        "Condition"
    )

    print(
        df.to_string()
    )

    display(df)

else:

    print(
        "No evaluation results found."
    )